# NBA Analytics Deep Dive

Advanced analytics using the nbadb star schema: lineup analysis, shot charts,
win probability curves, rolling averages, and on/off court impact.

In [ ]:
import duckdb
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

conn = duckdb.connect("../nbadb/nba.duckdb", read_only=True)
print("Connected to nba.duckdb")

In [ ]:
# Top 5-man lineups by net rating (min 100 minutes)
lineups = conn.sql("""
    SELECT
        group_name,
        min AS minutes,
        off_rating,
        def_rating,
        net_rating,
        gp
    FROM fact_lineup_stats
    WHERE season_id = '2024-25'
      AND group_quantity = 5
      AND min >= 100
    ORDER BY net_rating DESC
    LIMIT 20
""").pl()

print(f"Top lineups: {len(lineups)}")
lineups

In [ ]:
# Shot chart heatmap
shots = conn.sql("""
    SELECT loc_x, loc_y, shot_made_flag
    FROM fact_shot_chart
    WHERE season_id = '2024-25'
      AND loc_x IS NOT NULL
      AND loc_y IS NOT NULL
    LIMIT 50000
""").pl()

fig, ax = plt.subplots(figsize=(10, 9))

# Draw court outline
court = patches.Rectangle((-250, -47.5), 500, 470, linewidth=1,
                           edgecolor="black", facecolor="#f0f0f0")
ax.add_patch(court)
hoop = plt.Circle((0, 0), 7.5, linewidth=1, edgecolor="orange",
                    facecolor="none")
ax.add_patch(hoop)
three_arc = patches.Arc((0, 0), 475, 475, angle=0, theta1=22, theta2=158,
                         linewidth=1, edgecolor="gray")
ax.add_patch(three_arc)

ax.hexbin(
    shots["loc_x"].to_numpy(),
    shots["loc_y"].to_numpy(),
    C=shots["shot_made_flag"].to_numpy(),
    reduce_C_function=np.mean,
    gridsize=30, cmap="RdYlGn", mincnt=5, extent=(-250, 250, -50, 420)
)
plt.colorbar(ax.collections[-1], ax=ax, label="FG%")
ax.set_xlim(-260, 260)
ax.set_ylim(-55, 425)
ax.set_aspect("equal")
ax.set_title("Shot Chart Heatmap (2024-25)")
plt.tight_layout()
plt.show()

In [ ]:
# Win probability curve for a specific game
wp = conn.sql("""
    SELECT
        game_id,
        period,
        seconds_remaining,
        home_win_pct
    FROM fact_win_probability
    WHERE game_id = (
        SELECT game_id
        FROM fact_win_probability
        GROUP BY game_id
        ORDER BY STDDEV(home_win_pct) DESC
        LIMIT 1
    )
    ORDER BY period, seconds_remaining DESC
""").pl()

if len(wp) > 0:
    wp = wp.with_row_index("event_idx")
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(wp["event_idx"].to_numpy(), wp["home_win_pct"].to_numpy(),
            color="steelblue", linewidth=1)
    ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)
    ax.fill_between(
        wp["event_idx"].to_numpy(),
        wp["home_win_pct"].to_numpy(),
        0.5, alpha=0.15, color="steelblue"
    )
    ax.set_xlabel("Game Event")
    ax.set_ylabel("Home Win Probability")
    ax.set_title(f"Win Probability Curve: Game {wp['game_id'][0]}")
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()
else:
    print("No win probability data found.")

In [ ]:
# Player rolling averages (example: top scorer)
rolling = conn.sql("""
    WITH top_player AS (
        SELECT player_id
        FROM analytics_player_season_complete
        WHERE season_id = '2024-25'
        ORDER BY pts DESC
        LIMIT 1
    )
    SELECT
        r.game_date,
        p.player_name,
        r.pts_rolling_10,
        r.ast_rolling_10,
        r.reb_rolling_10
    FROM agg_player_rolling r
    JOIN dim_player p ON r.player_id = p.player_id
    WHERE r.player_id = (SELECT player_id FROM top_player)
      AND r.season_id = '2024-25'
    ORDER BY r.game_date
""").pl()

if len(rolling) > 0:
    player_name = rolling["player_name"][0]
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(range(len(rolling)), rolling["pts_rolling_10"].to_numpy(),
            label="PTS (10g)", linewidth=2)
    ax.plot(range(len(rolling)), rolling["ast_rolling_10"].to_numpy(),
            label="AST (10g)", linewidth=2)
    ax.plot(range(len(rolling)), rolling["reb_rolling_10"].to_numpy(),
            label="REB (10g)", linewidth=2)
    ax.set_xlabel("Game Number")
    ax.set_ylabel("Rolling Average (10 games)")
    ax.set_title(f"{player_name} Rolling Averages (2024-25)")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No rolling average data found.")

In [ ]:
# On/off court impact: top players by on-off net rating differential
on_off = conn.sql("""
    SELECT
        p.player_name,
        s.on_court_net_rating,
        s.off_court_net_rating,
        (s.on_court_net_rating - s.off_court_net_rating) AS on_off_diff,
        s.on_court_min
    FROM agg_on_off_splits s
    JOIN dim_player p ON s.player_id = p.player_id
    WHERE s.season_id = '2024-25'
      AND s.on_court_min >= 500
    ORDER BY on_off_diff DESC
    LIMIT 20
""").pl()

if len(on_off) > 0:
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = ["green" if v > 0 else "red" for v in on_off["on_off_diff"].to_list()]
    ax.barh(
        on_off["player_name"].to_list()[::-1],
        on_off["on_off_diff"].to_list()[::-1],
        color=colors[::-1],
        alpha=0.7,
    )
    ax.axvline(x=0, color="gray", linestyle="--")
    ax.set_xlabel("On/Off Net Rating Differential")
    ax.set_title("On/Off Court Impact (2024-25, min 500 min)")
    plt.tight_layout()
    plt.show()
else:
    print("No on/off splits data found.")

In [ ]:
conn.close()
print("Analysis complete!")

## Methodology Notes

- **Offensive/Defensive Rating**: Points scored/allowed per 100 possessions
- **Net Rating**: Offensive rating minus defensive rating
- **On/Off Splits**: Team net rating when a player is on court vs off court
- **Win Probability**: Model based on score margin, time remaining, and possession
- **Rolling Averages**: 10-game moving average for smoothing game-to-game variance
- **Shot Chart FG%**: Field goal percentage computed per hex bin (min 5 attempts)

All data sourced from [stats.nba.com](https://stats.nba.com) via the nbadb pipeline.
Schema documentation available at [nbadb.dev](https://nbadb.dev).